In [40]:
from langgraph.graph import StateGraph , START , END
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint

from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict

In [41]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    provider="auto",
    temperature=0.7,
    max_new_tokens=100000
)

model = ChatHuggingFace(llm = llm)

In [42]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    score:int

In [43]:
def create_outline(state:BlogState)->BlogState:
    title = state['title']

    prompt = f'Generate a outline on the given title {title}.'

    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state


In [44]:
def create_blog(state:BlogState)->BlogState:
    outline = state['outline']

    prompt = f'Create a blog on the given outline {outline}.'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [49]:
def evluate(state:BlogState)->BlogState:
    outline = state['outline']
    content = state['content']

    prompt = f'Give an integer score for the content {content} for given outline {outline}.'

    score =  model.invoke(prompt).content
    state['score'] = score

    return state

In [50]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('outline',create_outline)
graph.add_node('content',create_blog)
graph.add_node('evaluation',evluate)

#edges
graph.add_edge(START,'outline')
graph.add_edge('outline','content')
graph.add_edge('content','evaluation')
graph.add_edge('evaluation',END)

#compile
workflow = graph.compile()

In [51]:
initial_state = {'title':'Rise of AI'}
final_state = workflow.invoke(initial_state)

print(final_state['score'])
print(final_state['title'])
print(final_state['outline'])
print(final_state['content'])


**Score: 9 / 10**

**Why this rating**

| Criterion | Assessment (out of 10) | Comments |
|-----------|------------------------|----------|
| **Structure & Organization** | 10 | The piece follows the supplied outline perfectly, with clear headings, sub‑headings, and logical flow from history to future. |
| **Depth & Accuracy of Content** | 9 | Provides solid, up‑to‑date facts (e.g., AI‑Act 2024, GPT‑4, Gemini‑1) and covers the major technical, economic, and societal dimensions. Minor omissions (e.g., limited discussion of AI in finance) keep it from a perfect score. |
| **Clarity & Readability** | 9 | Language is engaging (hook, bullet tables) and jargon is explained; a few long sentences could be split for smoother reading. |
| **Use of Supporting Elements** | 9 | Tables, timelines, and a glossary add value; the “Optional Appendices” give flexibility for longer formats. |
| **Originality & Insight** | 8 | The article synthesizes known information well and offers thoughtful “Call‑to‑Ac